In [2]:
# methods: cleaning

import pandas as pd

files = {"income_downpayment": r"C:\sta141B\Metro Homeowner Income Downpayment Data.csv",
    "housing_growth": r"C:\sta141B\Metro Housing Data SFR Condo Growth.csv",
    "zhvi_monthly_growth": r"C:\sta141B\Metro_zhvi_uc_sfrcondo_tier_0.33_0.67_sm_sa_month.csv"}

# read csv data, remove extra spaces, divide columns as identiy columns and date dcolumns
def clean_csv(path):
    df = pd.read_csv(path)
    df.columns = df.columns.str.strip()
    identity_columns = [c for c in ["RegionID","SizeRank","RegionName","RegionType","StateName","BaseDate"] if c in df.columns]
    date_columns = [c for c in df.columns if c not in identity_columns]

    # convert to date
    if "BaseDate" in df.columns:
        df["BaseDate"] = pd.to_datetime(df["BaseDate"], errors="coerce")

    # convert date columns as numerical
    for c in date_columns:
        try:
            pd.to_datetime(c)
            df[c] = pd.to_numeric(df[c], errors="coerce")
        except:
            pass

    # fill blank name, remove duplicate data, and sorting tables
    df["StateName"] = df.get("StateName", pd.Series(index=df.index)).fillna("US")
    df = df.drop_duplicates().sort_values(["SizeRank","RegionName"], na_position="last").reset_index(drop=True)
    return df
cleaned = {name: clean_csv(path) for name, path in files.items()}

# three tables after cleanning
income_downpayment = cleaned["income_downpayment"]
housing_growth    = cleaned["housing_growth"]
zhvi_monthly_growth      = cleaned["zhvi_monthly_growth"]

# list first 5 regions and first 15 columns(include region informaion and 10 dates)
print("income_downpayment: ")
print(income_downpayment.iloc[:5, :15], "\n\n\n")
print("housing_growth")
print(housing_growth.iloc[:5, :15], "\n\n\n")
print("Zillow Home Value Index monthly growth")
print(zhvi_monthly_growth.iloc[:5, :15])

income_downpayment: 
   RegionID  SizeRank       RegionName RegionType StateName    1/31/2012  \
0    394913         1     New York, NY        msa        NY  95223.26513   
1    753899         2  Los Angeles, CA        msa        CA  78468.30133   
2    394463         3      Chicago, IL        msa        IL  45758.33846   
3    394514         4       Dallas, TX        msa        TX  38053.85738   
4    394692         5      Houston, TX        msa        TX  44076.92592   

     2/29/2012    3/31/2012    4/30/2012    5/31/2012    6/30/2012  \
0  94852.48950  95283.05125  95118.71688  94575.49198  93872.46048   
1  77983.15832  77899.25330  77145.37762  76190.77310  75470.48430   
2  45356.90329  45356.45295  45046.96589  44566.05970  44165.35852   
3  38026.34014  38301.18680  38335.69732  38178.47162  38026.17937   
4  44076.70738  44314.70179  44236.60787  43961.04461  43712.73057   

     7/31/2012    8/31/2012    9/30/2012   10/31/2012  
0  93130.62978  93470.46137  92814.13318  920

In [3]:
# methods: transformation

def transform(df):
    # convert those date columns, but reamin other types of information like RegionName as original 
    identity_columns = [c for c in ["RegionID","SizeRank","RegionName","RegionType","StateName","BaseDate"] if c in df.columns]
    df = df.melt(id_vars=identity_columns, var_name="Date", value_name="Value")
    df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
    df["Value"] = pd.to_numeric(df["Value"], errors="coerce")
    # remove unavailable dates
    df = df.dropna(subset=["Date"]).sort_values(["RegionName","Date"])
    # calculate previous hosuing value, current gousing value, and how much they surge up
    df["PrevValue"] = df.groupby("RegionName")["Value"].shift(1)
    df["Change"] = df["Value"] - df["PrevValue"]
    df["PctChange"] = df["Change"] / df["PrevValue"]
    return df.reset_index(drop=True)

income_downpayment_tr = transform(income_downpayment)
zhvi_monthly_growth_tr   = transform(zhvi_monthly_growth)
# tail(1) chooses the latest information
latest_income_downpayment = income_downpayment_tr.groupby("RegionName", as_index=False).tail(1)
latest_zhvi_index   = zhvi_monthly_growth_tr.groupby("RegionName", as_index=False).tail(1)

print("latest Income downpayment:")
print(latest_income_downpayment)
print("\n\n\n", "latest Zilow Home Value Index:")
print(latest_zhvi_index)

latest Income downpayment:
       RegionID  SizeRank      RegionName RegionType StateName       Date  \
168      394299       251     Abilene, TX        msa        TX 2026-01-31   
337      394304        83       Akron, OH        msa        OH 2026-01-31   
506      394306       295      Albany, GA        msa        GA 2026-01-31   
675      394308        64      Albany, NY        msa        NY 2026-01-31   
844      394307       330      Albany, OR        msa        OR 2026-01-31   
...         ...       ...             ...        ...       ...        ...   
65064    395240       193      Yakima, WA        msa        WA 2026-01-31   
65233    395244       122        York, PA        msa        PA 2026-01-31   
65402    395245       106  Youngstown, OH        msa        OH 2026-01-31   
65571    395246       247   Yuba City, CA        msa        CA 2026-01-31   
65740    395247       225        Yuma, AZ        msa        AZ 2026-01-31   

              Value     PrevValue     Change  Pc

In [1]:
# feature engineering 

import pandas as pd

files = {"income_downpayment": r"C:\sta141B\Metro Homeowner Income Downpayment Data.csv",
    "housing_growth": r"C:\sta141B\Metro Housing Data SFR Condo Growth.csv",
    "zhvi_monthly_growth": r"C:\sta141B\Metro_zhvi_uc_sfrcondo_tier_0.33_0.67_sm_sa_month.csv"}

def feature_engineering_part(feature_engineeering):
    df = pd.read_csv(feature_engineeering)
    # remove extra spaces
    df.columns = df.columns.str.strip()
    # select available identity columns
    identity_columns = [c for c in ["RegionID", "SizeRank", "RegionName", "RegionType", "StateName", "BaseDate"] if c in df.columns]
    # convert to long format, and then convert to correct date format
    df = df.melt(id_vars=identity_columns, var_name="Date", value_name="Value")
    df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
    df["Value"] = pd.to_numeric(df["Value"], errors="coerce")
    # remove unavailable columns and sort in order by time and cities
    df = df.dropna(subset=["Date"]).sort_values(["RegionName", "Date"]).reset_index(drop=True)
    group = df.groupby("RegionName")["Value"]
    df["Year"] = df["Date"].dt.year
    df["Month"] = df["Date"].dt.month
    # shifting to previous other time points
    df["Lag1"] = group.shift(1)
    df["Lag12"] = group.shift(12)
    df["MoM"] = (df["Value"] - df["Lag1"]) / df["Lag1"]
    df["YoY"] = (df["Value"] - df["Lag12"]) / df["Lag12"]

    return df

results = {name: feature_engineering_part(feature_engineeering) for name, feature_engineeering in files.items()}

income_downpayment_feature_engineering = results["income_downpayment"]
zhvi_monthly_growth_feature_engineering = results["zhvi_monthly_growth"]

print("Income Downpayment Feature Engineering:")
print(income_downpayment_feature_engineering.)
print("\n\n\n", "Zillow Home Value Index Growth(Variation) Feature Engineering:")
print(zhvi_monthly_growth_feature_engineering)


Income Downpayment Feature Engineering:
       RegionID  SizeRank   RegionName RegionType StateName       Date  \
0        394299       251  Abilene, TX        msa        TX 2012-01-31   
1        394299       251  Abilene, TX        msa        TX 2012-02-29   
2        394299       251  Abilene, TX        msa        TX 2012-03-31   
3        394299       251  Abilene, TX        msa        TX 2012-04-30   
4        394299       251  Abilene, TX        msa        TX 2012-05-31   
...         ...       ...          ...        ...       ...        ...   
65736    395247       225     Yuma, AZ        msa        AZ 2025-09-30   
65737    395247       225     Yuma, AZ        msa        AZ 2025-10-31   
65738    395247       225     Yuma, AZ        msa        AZ 2025-11-30   
65739    395247       225     Yuma, AZ        msa        AZ 2025-12-31   
65740    395247       225     Yuma, AZ        msa        AZ 2026-01-31   

             Value  Year  Month         Lag1        Lag12       MoM    

In [15]:
# method: acquisition

import pandas as pd

files = {
    "income_downpayment": r"C:\sta141B\Metro Homeowner Income Downpayment Data.csv",
    "housing_growth": r"C:\sta141B\Metro Housing Data SFR Condo Growth.csv",
    "zhvi_monthly_growth": r"C:\sta141B\Metro_zhvi_uc_sfrcondo_tier_0.33_0.67_sm_sa_month.csv"
}

income_downpayment_acqui = pd.read_csv(files["income_downpayment"])
housing_growth_acqui = pd.read_csv(files["housing_growth"])
zhvi_monthly_growth_acqui = pd.read_csv(files["zhvi_monthly_growth"])

print("income downpayment:")
print(income_downpayment_acqui)
print("\n\n\n","housing growth:")
print(housing_growth_acqui)
print("\n\n\n", "Zillow Housing Value Index monthly growth:")
print(zhvi_monthly_growth_acqui)

income downpayment:
     RegionID  SizeRank       RegionName RegionType StateName    1/31/2012  \
0      394913         1     New York, NY        msa        NY  95223.26513   
1      753899         2  Los Angeles, CA        msa        CA  78468.30133   
2      394463         3      Chicago, IL        msa        IL  45758.33846   
3      394514         4       Dallas, TX        msa        TX  38053.85738   
4      394692         5      Houston, TX        msa        TX  44076.92592   
..        ...       ...              ...        ...       ...          ...   
384    394787       518     Lewiston, ID        msa        ID  34582.28791   
385    394570       523         Enid, OK        msa        OK  19909.68154   
386    395199       525  Walla Walla, WA        msa        WA  38207.70803   
387    394444       551  Carson City, NV        msa        NV  31013.68513   
388    394551       552   Eagle Pass, TX        msa        TX  25970.13421   

       2/29/2012    3/31/2012    4/30/2012 